Objectives:
1. Label the processed training data with the levels provided by https://tocfl.edu.tw/tocfl/index.php/exam/download$0

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
import ast

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
TRAINING_DATA_PATH = '/content/drive/MyDrive/期末專題/processed_segments_with_ckip_文本.csv'
LEVEL_CSV_PATH = '/content/drive/MyDrive/期末專題/級數資料/漢字與詞語.csv'

In [ ]:
print("Loading datasets...")
df_train = pd.read_csv(TRAINING_DATA_PATH)
df_level = pd.read_csv(LEVEL_CSV_PATH)

Loading datasets...


# Input the rules of the leveling process.


1. Level X -> 單字 80% 在 X 範圍內
2. Level X+1 -> 允許 20% 左右 > level X的詞語



In [ ]:
word_level_dict = {}
if '漢字' in df_level.columns and '級別' in df_level.columns:
    dict1 = df_level.dropna(subset=['漢字', '級別']).set_index('漢字')['級別'].to_dict()
    word_level_dict.update(dict1)

print(f"Total unique words in level dictionary: {len(word_level_dict)}")

Total unique words in level dictionary: 16356


# Calculate the level

In [ ]:
def calculate_integer_weighted_level(word_list):
    if not isinstance(word_list, list) or len(word_list) == 0:
        return np.nan

    total_score = 0.0
    valid_word_count = 0 # 這裡的 N 只會計算「字典查得到」的詞

    for word in word_list:
        if word.strip() == "":
            continue

        if word in word_level_dict:
            valid_word_count += 1
            total_score += float(word_level_dict[word])

    if valid_word_count == 0:
        return np.nan

    average_score = total_score / valid_word_count
    final_int_level = int(round(average_score))
    final_int_level = max(1, min(final_int_level, 7))

    return final_int_level

In [ ]:
if 'ckip_ws_joined' in df_train.columns:
    df_train['word_list'] = df_train['ckip_ws_joined'].astype(str).str.split()
elif 'ckip_ws' in df_train.columns:
    def parse_ckip_ws(x):
        try: return ast.literal_eval(x) if isinstance(x, str) else x
        except: return str(x).split()
    df_train['word_list'] = df_train['ckip_ws'].apply(parse_ckip_ws)
else:
    raise ValueError("找不到 'ckip_ws_joined' 或 'ckip_ws' 欄位，請檢查資料結構！")

In [ ]:
df_train['calculated_level'] = df_train['word_list'].apply(calculate_integer_weighted_level)
df_train = df_train.drop(columns=['word_list'])

# Save the labeled training data

Process the final output so that the column will be text_segment and calculated_level

In [ ]:
df_output = df_train[['text_segment', 'calculated_level']]

OUTPUT_PATH = '/content/drive/MyDrive/期末專題/訓練資料文本/labeled_training_data.csv'
df_output.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')

print(f"\n計算完成！已篩選指定欄位並儲存至: {OUTPUT_PATH}")
print("\n--- 輸出資料前 5 筆預覽 ---")
print(df_output.head())

print("\n--- 各純整數級別文本數量統計 ---")
print(df_output['calculated_level'].value_counts().sort_index())


計算完成！已篩選指定欄位並儲存至: /content/drive/MyDrive/期末專題/訓練資料文本/labeled_training_data.csv

--- 輸出資料前 5 筆預覽 ---
                                        text_segment  calculated_level
0  這個小小的第三者，似乎一生下來就得到父母的鍾愛，在她噘著小嘴唇甜蜜睡覺的時候，在她睜開烏黑的...               2.0
1  我們的臥室開始有釘鎚的響聲，鐵絲安裝起來了，一道，兩道，三道，四道，五道，六道。她的尿布像一...               2.0
2  兩部車子回家的公務員生活樂趣被破壞了，但是我們卻從另一方面得到了補償。我們可以捏捏嬰兒的小手...               2.0
3  我們享受她給我們的一切聲音，這聲音使我們的房間格外溫暖。我們偷看她安靜時候臉上的表情，這表情...               2.0
4  小太陽不怕天上雲朵的遮掩，小太陽能透過雨絲，透過尿布的迷魂陣，透過愁苦靈魂堅硬的外殻，暖烘烘...               2.0

--- 各級別文本數量統計 ---
calculated_level
1.0      3
2.0    269
Name: count, dtype: int64
